# UQ in Post-Generation Attribution — Full Pipeline Notebook

This notebook mirrors `main.py` exactly, broken into clearly labelled cells.
Run cells **top to bottom**. Each cell corresponds to one pipeline step.

```
Step 1  → Install & Setup
Step 2  → API Key & Config
Step 3  → Build BM25 Index
Step 4  → Load NLI Model
Step 5  → Generate LLM Answer
Step 6  → Extract Atomic Claims
Step 7  → Generate 12 Perturbations
Step 8  → Retrieve Docs + NLI Score (all variants)
Step 9  → UQ1: Attribution Uncertainty
Step 10 → UQ2: Retrieval Uncertainty
Step 11 → UQ3: Content / Hallucination Uncertainty
Step 12 → UQ4: Claim Extraction Diagnostic
Step 13 → Composite U_attr Score
Step 14 → Full Experiment Loop (all questions)
Step 15 → Save Results & Report
```

## Step 1 — Install Dependencies

In [ ]:
import subprocess, sys
pkgs = ['openai','python-dotenv','sentence-transformers','rank-bm25',
        'datasets','tqdm','transformers','torch','numpy','pandas','scikit-learn']
for pkg in pkgs:
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg], capture_output=True)
    print(f"{'OK' if r.returncode==0 else 'FAIL'}: {pkg}")

## Step 2 — API Key & Configuration

In [ ]:
import os, sys, getpass
from pathlib import Path

# ── Add pipeline directory to path so imports work ──────────────────────────
# If running from repo root: uncomment the next line and adjust path
# sys.path.insert(0, str(Path('.')))

# ── Load API key ─────────────────────────────────────────────────────────────
def _load_key():
    env = Path('.env')
    if env.exists():
        for line in env.read_text().splitlines():
            if line.startswith('OPENROUTER_API_KEY='):
                k = line.split('=',1)[1].strip().strip('"').strip("'")
                if k: return k
    k = os.environ.get('OPENROUTER_API_KEY','')
    if k: return k
    try:
        from google.colab import userdata
        k = userdata.get('OPENROUTER_API_KEY')
        if k: return k
    except: pass
    print('API key not found. Enter below:')
    return getpass.getpass('OPENROUTER_API_KEY: ').strip()

OPENROUTER_API_KEY = _load_key()
assert OPENROUTER_API_KEY, 'Empty API key'
print(f'API key loaded ({len(OPENROUTER_API_KEY)} chars)')

# ── Configuration ─────────────────────────────────────────────────────────────
import numpy as np
import random
NUM_QUESTIONS = 3      # ← change to 5 or 17 for full run
TOP_K         = 5      # BM25 top-k docs to retrieve
SEED          = 42

# Weights for composite U_attr = w1*UQ1 + w2*UQ2 + w3*UQ3
# Set w3=0 to skip content UQ entirely
W1 = 0.40   # UQ1 Attribution
W2 = 0.35   # UQ2 Retrieval
W3 = 0.25   # UQ3 Content (set to 0 to skip)

assert abs(W1+W2+W3 - 1.0) < 1e-6, f'Weights must sum to 1.0 (got {W1+W2+W3})'
random.seed(SEED); np.random.seed(SEED)
print(f'Config: {NUM_QUESTIONS} questions | top_k={TOP_K} | weights: {W1}/{W2}/{W3}')

# ── Results directory ─────────────────────────────────────────────────────────
from pathlib import Path
RESULTS_DIR = Path('results'); RESULTS_DIR.mkdir(exist_ok=True)

## Step 3 — Build BM25 Index (ASQA + ALCE corpus)

In [ ]:
from pipeline_utils import build_bm25_index, bm25_retrieve

# Builds BM25 index over ASQA annotations + ALCE Wikipedia passages
# Returns: questions list, bm25 index, corpus_texts, corpus_ids
questions, bm25_index, corpus_texts, corpus_ids = build_bm25_index(
    num_questions=NUM_QUESTIONS
)

# Quick test: retrieve docs for first question
test_docs = bm25_retrieve(questions[0], bm25_index, corpus_texts, corpus_ids, top_k=3)
print(f'Test retrieval for: "{questions[0][:60]}"')
for d in test_docs:
    print(f'  [score={d["score"]:.2f}] {d["text"][:80]}')
print(f'\nbm25_retrieve() is ready.')

## Step 4 — Load NLI Model

In [ ]:
from pipeline_utils import load_nli_model, nli_score_claim

# cross-encoder/nli-deberta-v3-base
# Labels: [contradiction=0, entailment=1, neutral=2]
# We use column index 1 (entailment probability) as attribution score.
nli_model = load_nli_model()

# Quick NLI test
test_claim = 'Water boils at 100 degrees Celsius at sea level.'
test_d = [{'text':'Water has a boiling point of 100°C at standard pressure.','score':10,'id':'t1'},
          {'text':'The capital of France is Paris.','score':3,'id':'t2'}]
r = nli_score_claim(test_claim, test_d, nli_model)
print(f'NLI test per_doc entailment: {[round(x,3) for x in r["per_doc_entailment"]]}')
print(f'max_score: {r["max_score"]:.3f}  (doc1 should be high, doc2 low)')

## Step 5 — Build LLM Client & Generate Answer for One Question

In [ ]:
from pipeline_utils import build_llm_client, generate_answer

llm    = build_llm_client(OPENROUTER_API_KEY)
q_demo = questions[0]

print(f'Question: {q_demo}')
answer = generate_answer(llm, q_demo)
print(f'\nAnswer:\n{answer}')

## Step 6 — Extract Atomic Claims

In [ ]:
from pipeline_utils import extract_atomic_claims

# temperature=0.0 for deterministic extraction in main run
claims = extract_atomic_claims(llm, answer, temperature=0.0)

print(f'{len(claims)} atomic claims extracted:')
for i, c in enumerate(claims, 1):
    print(f'  {i}. {c}')

## Step 7 — Generate 12 Perturbations for Claim #1

In [ ]:
from pipeline_utils import generate_all_perturbations

# Work with the first claim for demonstration
demo_claim = claims[0]
print(f'Base claim: {demo_claim}\n')

# Generates all 12 perturbations across 5 categories:
# C1 (×2 Preserve) + C2 (×4 Destroy) + C3 (×2 Specificity) + C4 (×2 Breadth) + C5 (×2 Depth)
perturbations = generate_all_perturbations(llm, demo_claim)

for key, text in perturbations.items():
    cat = key.split('_')[0]  # C1, C2, etc.
    print(f'  [{cat}] {key:<28} → {text[:80]}')

## Step 8 — Retrieve Docs + NLI Score for Base Claim + All 12 Perturbations

In [ ]:
# This produces the two maps that all UQ modules consume:
#   docs_map       : {variant_key → [doc dicts]}
#   nli_scores_map : {variant_key → nli_score_claim() output}

docs_map       = {}
nli_scores_map = {}

# Base claim
base_docs              = bm25_retrieve(demo_claim, bm25_index, corpus_texts, corpus_ids, TOP_K)
docs_map['base']       = base_docs
nli_scores_map['base'] = nli_score_claim(demo_claim, base_docs, nli_model)

# All 12 perturbations
from tqdm.auto import tqdm
for key, pert_text in tqdm(perturbations.items(), desc='Retrieving + NLI scoring'):
    p_docs              = bm25_retrieve(pert_text, bm25_index, corpus_texts, corpus_ids, TOP_K)
    docs_map[key]       = p_docs
    nli_scores_map[key] = nli_score_claim(pert_text, p_docs, nli_model)

print('\nNLI max_score per variant:')
for k, v in nli_scores_map.items():
    print(f'  {k:<30} → max={v["max_score"]:.3f}')

## Step 9 — UQ₁: Attribution Uncertainty

In [ ]:
from uq_attribution import compute_uq_attribution, interpret_uq1

# Computes NLI entailment variance across all 13 variants (base + 12 perturbs)
# UQ1_norm = Var(all_scores) / 0.25   (normalized to [0,1])
uq1 = compute_uq_attribution(nli_scores_map)

print('UQ₁ — Attribution Uncertainty')
print(f'  All 13 NLI scores : {[round(x,3) for x in uq1["scores_all"]]}')
print(f'  UQ1_raw (variance): {uq1["UQ1_raw"]:.5f}')
print(f'  UQ1_norm          : {uq1["UQ1_norm"]:.4f}  ← used in composite')
print(f'  Preserve C1 mean  : {uq1["score_preserve_mean"]:.3f}')
print(f'  Destroy  C2 mean  : {uq1["score_destroy_mean"]:.3f}')
print(f'\n  {interpret_uq1(uq1)}')

## Step 10 — UQ₂: Retrieval Uncertainty (RBO + Jaccard)

In [ ]:
from uq_retrieval import compute_uq_retrieval, interpret_uq2

# Primary signal: Instability = 1 - mean(RBO over 12 perturbation queries)
# Jaccard is stored as diagnostic only.
uq2 = compute_uq_retrieval(docs_map)

print('UQ₂ — Retrieval Uncertainty')
print(f'  Mean RBO     : {uq2["mean_rbo"]:.4f}')
print(f'  Mean Jaccard : {uq2["mean_jaccard"]:.4f}  (diagnostic only)')
print(f'  UQ2_norm     : {uq2["UQ2_norm"]:.4f}  ← Instability = 1 - mean_RBO')
print()
print('  Per-perturbation RBO scores:')
for k, rbo in sorted(uq2['rbo_scores'].items(), key=lambda x: x[1]):
    print(f'    {k:<30} RBO={rbo:.3f} | Jaccard={uq2["jaccard_scores"][k]:.3f}')
print(f'\n  {interpret_uq2(uq2)}')

## Step 11 — UQ₃: Hallucination / Content Uncertainty

In [ ]:
# UQ3 = avg(Signal_A, Signal_B)
# Signal A: Retrieval Anchoring Bias  = Score_base - mean(all 12 perturb scores)
# Signal B: Specification Entailment Gap = Score_base - mean(C5_causal, C5_consequence)

# Skip if W3 = 0
if W3 > 0:
    from uq_content import compute_uq_content, interpret_uq3
    uq3 = compute_uq_content(nli_scores_map)

    print('UQ₃ — Hallucination / Content Uncertainty')
    print(f'  Score_base          : {uq3["score_base"]:.3f}')
    print(f'  Signal_A (Drop)     : raw={uq3["Signal_A_raw"]:.3f} | norm={uq3["Signal_A_norm"]:.3f}')
    print(f'  Signal_B (SEG)      : raw={uq3["Signal_B_raw"]:.3f} | norm={uq3["Signal_B_norm"]:.3f}')
    print(f'  C5 specification scores: {uq3["c5_scores"]}')
    print(f'  UQ3_norm            : {uq3["UQ3_norm"]:.4f}  ← used in composite')
    print(f'\n  {interpret_uq3(uq3)}')
else:
    uq3 = None
    print('UQ₃ skipped (W3=0)')

## Step 12 — UQ₄: Claim Extraction Diagnostic (optional)

In [ ]:
# UQ4 is a DIAGNOSTIC — it does NOT enter the composite U_attr.
# Runs claim extraction 3 times at temperature>0 and measures NLI agreement.
# Uncomment to run (adds ~3x the LLM calls for extraction):

# from uq_claim_extraction import compute_uq_claim_extraction
# uq4 = compute_uq_claim_extraction(llm, answer, nli_model, n_runs=3, temperature=0.7)
# print('UQ₄ — Claim Extraction Diagnostic (not in composite)')
# print(uq4['interpretation'])
# print(f'  Claim counts per run: {uq4["claim_counts"]}')
# print(f'  Count variance      : {uq4["count_variance"]:.2f}')
# print(f'  Mean NLI agreement  : {uq4["mean_agreement"]:.3f}')

print('UQ₄ cell ready — uncomment above to run.')

## Step 13 — Composite U_attr Score

In [ ]:
# U_attr = W1*UQ1 + W2*UQ2 + W3*UQ3
# If W3=0, UQ3 contributes 0.

uq3_norm = uq3['UQ3_norm'] if uq3 else 0.0
u_attr   = W1 * uq1['UQ1_norm'] + W2 * uq2['UQ2_norm'] + W3 * uq3_norm
level    = 'HIGH' if u_attr >= 0.5 else ('MODERATE' if u_attr >= 0.2 else 'STABLE')

print('=' * 55)
print(f'  Claim  : {demo_claim[:60]}')
print(f'  UQ1    : {uq1["UQ1_norm"]:.4f}  (weight={W1})')
print(f'  UQ2    : {uq2["UQ2_norm"]:.4f}  (weight={W2})')
print(f'  UQ3    : {uq3_norm:.4f}  (weight={W3})')
print(f'  ─────────────────────────────────────')
print(f'  U_attr : {u_attr:.4f}  [{level}]')
print('=' * 55)

## Step 14 — Full Experiment Loop (all questions)

In [ ]:
import json
from tqdm.auto import tqdm
from pipeline_utils import generate_all_perturbations, ALL_PERT_KEYS
from uq_attribution  import compute_uq_attribution
from uq_retrieval    import compute_uq_retrieval
from uq_content      import compute_uq_content

all_records = []
JSONL_PATH  = RESULTS_DIR / 'attribution_uncertainty.jsonl'
if JSONL_PATH.exists(): JSONL_PATH.unlink()

for q_idx, question in tqdm(enumerate(questions), total=len(questions), desc='Questions'):
    print(f'\nQ-{q_idx+1}: {question}')

    answer = generate_answer(llm, question)
    if not answer: continue

    claims = extract_atomic_claims(llm, answer, temperature=0.0)
    if not claims: continue
    print(f'  {len(claims)} claims extracted')

    for c_idx, claim in tqdm(enumerate(claims), total=len(claims), leave=False):

        perturbations = generate_all_perturbations(llm, claim)

        # Retrieve + NLI for base + all 12 perturbations
        docs_map, nli_scores_map = {}, {}
        for key, text in [('base', claim)] + list(perturbations.items()):
            docs = bm25_retrieve(text, bm25_index, corpus_texts, corpus_ids, TOP_K)
            docs_map[key]       = docs
            nli_scores_map[key] = nli_score_claim(text, docs, nli_model)

        # Compute UQ signals
        uq1     = compute_uq_attribution(nli_scores_map)
        uq2     = compute_uq_retrieval(docs_map)
        uq3     = compute_uq_content(nli_scores_map) if W3 > 0 else None
        uq3_n   = uq3['UQ3_norm'] if uq3 else 0.0
        u_attr  = W1*uq1['UQ1_norm'] + W2*uq2['UQ2_norm'] + W3*uq3_n
        level   = 'HIGH' if u_attr >= 0.5 else ('MODERATE' if u_attr >= 0.2 else 'STABLE')

        record = {
            'q_idx':q_idx, 'question':question, 'answer':answer,
            'c_idx':c_idx, 'claim':claim,
            'UQ1_norm':uq1['UQ1_norm'], 'UQ2_norm':uq2['UQ2_norm'], 'UQ3_norm':uq3_n,
            'U_attr':u_attr, 'level':level,
            'mean_rbo':uq2['mean_rbo'], 'mean_jaccard':uq2['mean_jaccard'],
            'score_preserve_mean':uq1['score_preserve_mean'],
            'score_destroy_mean': uq1['score_destroy_mean'],
            'Signal_A_norm': uq3['Signal_A_norm'] if uq3 else None,
            'Signal_B_norm': uq3['Signal_B_norm'] if uq3 else None,
            'weights':{'w1':W1,'w2':W2,'w3':W3},
        }
        all_records.append(record)
        with open(JSONL_PATH,'a') as f:
            f.write(json.dumps(record)+'\n')

print(f'\nDone: {len(all_records)} claim records collected')

## Step 15 — Save Results & Print Report

In [ ]:
import pandas as pd

if not all_records:
    print('No records. Run Step 14 first.')
else:
    df = pd.DataFrame(all_records)
    df.to_csv(RESULTS_DIR / 'attribution_uncertainty.csv', index=False)

    # Per-question summary
    q_summary = []
    for qi, grp in df.groupby('q_idx'):
        q_summary.append({
            'q_idx': int(qi), 'question': grp['question'].iloc[0],
            'num_claims': len(grp),
            'mean_U_attr': float(grp['U_attr'].mean()),
            'mean_UQ1': float(grp['UQ1_norm'].mean()),
            'mean_UQ2': float(grp['UQ2_norm'].mean()),
            'mean_UQ3': float(grp['UQ3_norm'].mean()),
            'high_pct': float((grp['level']=='HIGH').mean()*100),
        })
    with open(RESULTS_DIR/'question_summary.json','w') as f:
        json.dump(q_summary, f, indent=2)

    # Report
    print('='*60)
    print('  RESULTS')
    print(f'  Total claims : {len(df)}')
    print(f'  Mean U_attr  : {df["U_attr"].mean():.4f}')
    c = df['level'].value_counts()
    for l in ['STABLE','MODERATE','HIGH']:
        n=c.get(l,0); print(f'  {l:<10}: {n} ({n/len(df)*100:.1f}%)')
    print(f'  Mean UQ1: {df["UQ1_norm"].mean():.4f} | UQ2: {df["UQ2_norm"].mean():.4f} | UQ3: {df["UQ3_norm"].mean():.4f}')
    print('='*60)
    print('Files saved to results/')

    display(df[['claim','UQ1_norm','UQ2_norm','UQ3_norm','U_attr','level']].head(10))

## Download Results (Colab)

In [ ]:
# Uncomment to download results from Colab:
# from google.colab import files
# files.download('results/attribution_uncertainty.csv')
# files.download('results/attribution_uncertainty.jsonl')
print('Files in results/:')
for p in sorted(Path('results').rglob('*')):
    if p.is_file(): print(f'  {p.name}  ({p.stat().st_size:,} bytes)')